In [4]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Offline Arabic ADAS.coco.zip"
extract_path = "/content/adas_coco"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted to:", extract_path)

Extracted to: /content/adas_coco


In [6]:
import os

for root, dirs, files in os.walk("/content/adas_coco"):
    for file in files:
        if file.endswith(".json"):
            print(os.path.join(root, file))

/content/adas_coco/valid/_annotations.coco.json
/content/adas_coco/test/_annotations.coco.json
/content/adas_coco/train/_annotations.coco.json


In [7]:
!find /content/adas_coco -maxdepth 2 -type d

/content/adas_coco
/content/adas_coco/valid
/content/adas_coco/test
/content/adas_coco/train


In [8]:
import json

annotation_path = "/content/adas_coco/train/_annotations.coco.json"

with open(annotation_path, "r", encoding="utf-8") as f:
    coco = json.load(f)

print("Images:", len(coco["images"]))
print("Annotations:", len(coco["annotations"]))
print("Classes:", len(coco["categories"]))

for category in coco["categories"]:
    print(f'{category["id"]}: {category["name"]}')

Images: 6191
Annotations: 8520
Classes: 15
0: ADAS-Objects
1: loose_gravel
2: no_entry
3: no_parking_waiting
4: pedestrian
5: pedestrian_crossing_sign
6: speed_100
7: speed_30
8: speed_40
9: speed_50
10: speed_60
11: speed_80
12: speed_bump
13: stoop_sign
14: traffic_signal


In [9]:
from collections import Counter
import json

annotation_path = "/content/adas_coco/train/_annotations.coco.json"

with open(annotation_path, "r", encoding="utf-8") as f:
    coco = json.load(f)

category_names = {
    category["id"]: category["name"]
    for category in coco["categories"]
}

counts = Counter(
    annotation["category_id"]
    for annotation in coco["annotations"]
)

for category_id, name in category_names.items():
    print(f"{category_id}: {name} -> {counts[category_id]} annotations")

0: ADAS-Objects -> 0 annotations
1: loose_gravel -> 299 annotations
2: no_entry -> 612 annotations
3: no_parking_waiting -> 312 annotations
4: pedestrian -> 4379 annotations
5: pedestrian_crossing_sign -> 239 annotations
6: speed_100 -> 197 annotations
7: speed_30 -> 309 annotations
8: speed_40 -> 282 annotations
9: speed_50 -> 244 annotations
10: speed_60 -> 283 annotations
11: speed_80 -> 243 annotations
12: speed_bump -> 134 annotations
13: stoop_sign -> 268 annotations
14: traffic_signal -> 719 annotations


**الخلية 1 — تجهيز RT-DETRv2**

In [10]:
# Clone the official RT-DETR repository
!git clone -q https://github.com/lyuwenyu/RT-DETR.git

# Move into the PyTorch implementation
%cd /content/RT-DETR/rtdetrv2_pytorch

# Install required dependencies
!pip install -q -r requirements.txt

# Verify the environment
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

/content/RT-DETR/rtdetrv2_pytorch
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.7/604.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.7/246.7 MB 5.3 MB/s eta 0:00:00
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA: 12.8


**الخلية 2 — التأكد من RT-DETRv2 قبل تحميل الـDataset**

In [11]:
import os

print("RT-DETRv2 directory:", os.getcwd())
print("Config directory exists:", os.path.exists("configs/rtdetrv2"))
print("Dataset config directory exists:", os.path.exists("configs/dataset"))

print("\nRT-DETRv2 configs:")
for file in os.listdir("configs/rtdetrv2"):
    print(" -", file)

print("\nDataset configs:")
for file in os.listdir("configs/dataset"):
    print(" -", file)

RT-DETRv2 directory: /content/RT-DETR/rtdetrv2_pytorch
Config directory exists: True
Dataset config directory exists: True

RT-DETRv2 configs:
 - rtdetrv2_hgnetv2_l_6x_coco.yml
 - rtdetrv2_r34vd_dsp_1x_coco.yml
 - rtdetrv2_r50vd_dsp_1x_coco.yml
 - rtdetrv2_hgnetv2_h_6x_coco.yml
 - rtdetrv2_r18vd_dsp_3x_coco.yml
 - rtdetrv2_r18vd_sp2_120e_coco.yml
 - rtdetrv2_r18vd_120e_coco.yml
 - rtdetrv2_r34vd_120e_coco.yml
 - rtdetrv2_r50vd_6x_coco.yml
 - rtdetrv2_r18vd_120e_voc.yml
 - rtdetrv2_hgnetv2_x_6x_coco.yml
 - rtdetrv2_r50vd_m_7x_coco.yml
 - rtdetrv2_r50vd_m_dsp_3x_coco.yml
 - rtdetrv2_r18vd_sp1_120e_coco.yml
 - rtdetrv2_r18vd_sp3_120e_coco.yml
 - rtdetrv2_r101vd_6x_coco.yml
 - include

Dataset configs:
 - voc_detection.yml
 - coco_detection.yml


**الخلية 3 — إنشاء Dataset Config**

In [12]:
import os

config_path = "configs/dataset/adas_detection.yml"

config = """task: detection

evaluator:
  type: CocoEvaluator
  iou_types: ['bbox']

num_classes: 14
remap_mscoco_category: False

train_dataloader:
  type: DataLoader
  dataset:
    type: CocoDetection
    img_folder: /content/adas_coco/train
    ann_file: /content/adas_coco/train/_annotations.coco.json
    return_masks: False
    transforms:
      type: Compose
      ops: ~
  shuffle: True
  num_workers: 4
  drop_last: True
  collate_fn:
    type: BatchImageCollateFunction

val_dataloader:
  type: DataLoader
  dataset:
    type: CocoDetection
    img_folder: /content/adas_coco/valid
    ann_file: /content/adas_coco/valid/_annotations.coco.json
    return_masks: False
    transforms:
      type: Compose
      ops: ~
  shuffle: False
  num_workers: 4
  drop_last: False
  collate_fn: BatchImageCollateFunction
"""

with open(config_path, "w", encoding="utf-8") as f:
    f.write(config)

print("Created:", config_path)
print("\n--- Config ---\n")

with open(config_path, "r", encoding="utf-8") as f:
    print(f.read())

Created: configs/dataset/adas_detection.yml

--- Config ---

task: detection

evaluator:
  type: CocoEvaluator
  iou_types: ['bbox']

num_classes: 14
remap_mscoco_category: False

train_dataloader:
  type: DataLoader
  dataset:
    type: CocoDetection
    img_folder: /content/adas_coco/train
    ann_file: /content/adas_coco/train/_annotations.coco.json
    return_masks: False
    transforms:
      type: Compose
      ops: ~
  shuffle: True
  num_workers: 4
  drop_last: True
  collate_fn:
    type: BatchImageCollateFunction

val_dataloader:
  type: DataLoader
  dataset:
    type: CocoDetection
    img_folder: /content/adas_coco/valid
    ann_file: /content/adas_coco/valid/_annotations.coco.json
    return_masks: False
    transforms:
      type: Compose
      ops: ~
  shuffle: False
  num_workers: 4
  drop_last: False
  collate_fn: BatchImageCollateFunction



**الخلية 4 — فحص الـCOCO loader والـcategory mapping**

In [13]:
import json

annotation_path = "/content/adas_coco/train/_annotations.coco.json"

with open(annotation_path, "r", encoding="utf-8") as f:
    coco = json.load(f)

print("Number of images:", len(coco["images"]))
print("Number of annotations:", len(coco["annotations"]))
print("Number of categories:", len(coco["categories"]))

print("\nCategories:")
for category in coco["categories"]:
    print(f"ID {category['id']}: {category['name']}")

annotation_category_ids = sorted(
    set(annotation["category_id"] for annotation in coco["annotations"])
)

print("\nCategory IDs used by annotations:")
print(annotation_category_ids)

print("\nMinimum category ID:", min(annotation_category_ids))
print("Maximum category ID:", max(annotation_category_ids))

Number of images: 6191
Number of annotations: 8520
Number of categories: 15

Categories:
ID 0: ADAS-Objects
ID 1: loose_gravel
ID 2: no_entry
ID 3: no_parking_waiting
ID 4: pedestrian
ID 5: pedestrian_crossing_sign
ID 6: speed_100
ID 7: speed_30
ID 8: speed_40
ID 9: speed_50
ID 10: speed_60
ID 11: speed_80
ID 12: speed_bump
ID 13: stoop_sign
ID 14: traffic_signal

Category IDs used by annotations:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

Minimum category ID: 1
Maximum category ID: 14


**الخلية 5 — العثور على الـCOCO Loader**

In [15]:
import os

matches = []

for root, dirs, files in os.walk("src"):
    for file in files:
        if file.endswith(".py"):
            path = os.path.join(root, file)

            try:
                with open(path, "r", encoding="utf-8") as f:
                    content = f.read()

                if "class CocoDetection" in content:
                    matches.append(path)

            except Exception:
                pass

print("Found CocoDetection in:")

for path in matches:
    print(path)

Found CocoDetection in:
src/data/dataset/coco_dataset.py


**الخلية 6 — فحص الـCocoDetection الحقيقي**

In [16]:
import inspect
from src.data.dataset.coco_dataset import CocoDetection

print(inspect.getsource(CocoDetection))

@register()
class CocoDetection(FasterCocoDetection, DetDataset):
    __inject__ = ['transforms', ]
    __share__ = ['remap_mscoco_category']
    
    def __init__(self, img_folder, ann_file, transforms, return_masks=False, remap_mscoco_category=False):
        super(FasterCocoDetection, self).__init__(img_folder, ann_file)
        self._transforms = transforms
        self.prepare = ConvertCocoPolysToMask(return_masks)
        self.img_folder = img_folder
        self.ann_file = ann_file
        self.return_masks = return_masks
        self.remap_mscoco_category = remap_mscoco_category

    def __getitem__(self, idx):
        img, target = self.load_item(idx)
        if self._transforms is not None:
            img, target, _ = self._transforms(img, target, self)
        return img, target

    def load_item(self, idx):
        image, target = super(FasterCocoDetection, self).__getitem__(idx)
        image_id = self.ids[idx]
        target = {'image_id': image_id, 'annotations': targe

**الخلية 7 — فحص تحويل COCO إلى labels**

In [17]:
import inspect
from src.data.dataset.coco_dataset import ConvertCocoPolysToMask

print(inspect.getsource(ConvertCocoPolysToMask))

class ConvertCocoPolysToMask(object):
    def __init__(self, return_masks=False):
        self.return_masks = return_masks

    def __call__(self, image: Image.Image, target, **kwargs):
        w, h = image.size

        image_id = target["image_id"]
        image_id = torch.tensor([image_id])

        anno = target["annotations"]

        anno = [obj for obj in anno if 'iscrowd' not in obj or obj['iscrowd'] == 0]

        boxes = [obj["bbox"] for obj in anno]
        # guard against no boxes via resizing
        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        boxes[:, 2:] += boxes[:, :2]
        boxes[:, 0::2].clamp_(min=0, max=w)
        boxes[:, 1::2].clamp_(min=0, max=h)

        category2label = kwargs.get('category2label', None)
        if category2label is not None:
            labels = [category2label[obj["category_id"]] for obj in anno]
        else:
            labels = [obj["category_id"] for obj in anno]
            
        labels = torch.tensor(

**الخلية 8 — إنشاء نسخة COCO نظيفة لـRT-DETR**

In [18]:
import json
import os
import shutil

source_root = "/content/adas_coco"
target_root = "/content/adas_coco_rtdetr"

splits = ["train", "valid", "test"]

category_mapping = {
    1: 0,
    2: 1,
    3: 2,
    4: 3,
    5: 4,
    6: 5,
    7: 6,
    8: 7,
    9: 8,
    10: 9,
    11: 10,
    12: 11,
    13: 12,
    14: 13,
}

new_categories = [
    {"id": 0, "name": "loose_gravel", "supercategory": "ADAS"},
    {"id": 1, "name": "no_entry", "supercategory": "ADAS"},
    {"id": 2, "name": "no_parking_waiting", "supercategory": "ADAS"},
    {"id": 3, "name": "pedestrian", "supercategory": "ADAS"},
    {"id": 4, "name": "pedestrian_crossing_sign", "supercategory": "ADAS"},
    {"id": 5, "name": "speed_100", "supercategory": "ADAS"},
    {"id": 6, "name": "speed_30", "supercategory": "ADAS"},
    {"id": 7, "name": "speed_40", "supercategory": "ADAS"},
    {"id": 8, "name": "speed_50", "supercategory": "ADAS"},
    {"id": 9, "name": "speed_60", "supercategory": "ADAS"},
    {"id": 10, "name": "speed_80", "supercategory": "ADAS"},
    {"id": 11, "name": "speed_bump", "supercategory": "ADAS"},
    {"id": 12, "name": "stoop_sign", "supercategory": "ADAS"},
    {"id": 13, "name": "traffic_signal", "supercategory": "ADAS"},
]

os.makedirs(target_root, exist_ok=True)

for split in splits:
    source_dir = os.path.join(source_root, split)
    target_dir = os.path.join(target_root, split)

    os.makedirs(target_dir, exist_ok=True)

    # Copy images without modifying them
    for filename in os.listdir(source_dir):
        if filename.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
            shutil.copy2(
                os.path.join(source_dir, filename),
                os.path.join(target_dir, filename)
            )

    source_json = os.path.join(source_dir, "_annotations.coco.json")
    target_json = os.path.join(target_dir, "_annotations.coco.json")

    with open(source_json, "r", encoding="utf-8") as f:
        coco = json.load(f)

    # Remap annotation category IDs
    for annotation in coco["annotations"]:
        old_id = annotation["category_id"]

        if old_id not in category_mapping:
            raise ValueError(
                f"Unexpected category_id {old_id} found in {split}"
            )

        annotation["category_id"] = category_mapping[old_id]

    # Replace categories with exactly 14 valid classes
    coco["categories"] = new_categories

    with open(target_json, "w", encoding="utf-8") as f:
        json.dump(coco, f, ensure_ascii=False)

    print(
        f"{split}: {len(coco['images'])} images, "
        f"{len(coco['annotations'])} annotations"
    )

print("\nCreated:", target_root)

train: 6191 images, 8520 annotations
valid: 1767 images, 2405 annotations
test: 884 images, 1278 annotations

Created: /content/adas_coco_rtdetr


**الخلية 9 — التحقق النهائي من الـclass mapping**

In [19]:
import json
from collections import Counter

annotation_path = "/content/adas_coco_rtdetr/train/_annotations.coco.json"

with open(annotation_path, "r", encoding="utf-8") as f:
    coco = json.load(f)

category_names = {
    category["id"]: category["name"]
    for category in coco["categories"]
}

counts = Counter(
    annotation["category_id"]
    for annotation in coco["annotations"]
)

print("Categories:")
for category_id in sorted(category_names):
    print(
        f"{category_id}: "
        f"{category_names[category_id]} -> "
        f"{counts[category_id]} annotations"
    )

used_ids = sorted(counts.keys())

print("\nCategory IDs used:")
print(used_ids)

print("\nNumber of classes:", len(category_names))
print("Number of used classes:", len(used_ids))
print("Total annotations:", sum(counts.values()))

assert used_ids == list(range(14))
assert len(category_names) == 14
assert sum(counts.values()) == 8520

print("\n✓ COCO dataset mapping is valid for 14 classes.")

Categories:
0: loose_gravel -> 299 annotations
1: no_entry -> 612 annotations
2: no_parking_waiting -> 312 annotations
3: pedestrian -> 4379 annotations
4: pedestrian_crossing_sign -> 239 annotations
5: speed_100 -> 197 annotations
6: speed_30 -> 309 annotations
7: speed_40 -> 282 annotations
8: speed_50 -> 244 annotations
9: speed_60 -> 283 annotations
10: speed_80 -> 243 annotations
11: speed_bump -> 134 annotations
12: stoop_sign -> 268 annotations
13: traffic_signal -> 719 annotations

Category IDs used:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]

Number of classes: 14
Number of used classes: 14
Total annotations: 8520

✓ COCO dataset mapping is valid for 14 classes.


**الخلية 10 — تحميل RT-DETRv2-S pretrained**

In [20]:
import os

checkpoint_dir = "/content/RT-DETR/checkpoints"
checkpoint_path = os.path.join(
    checkpoint_dir,
    "rtdetrv2_r18vd_120e_coco_rerun_48.1.pth"
)

os.makedirs(checkpoint_dir, exist_ok=True)

if not os.path.exists(checkpoint_path):
    !wget -q --show-progress \
        https://github.com/lyuwenyu/storage/releases/download/v0.2/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth \
        -O "{checkpoint_path}"

print("Checkpoint:", checkpoint_path)
print("Exists:", os.path.exists(checkpoint_path))

if os.path.exists(checkpoint_path):
    print("Size: {:.2f} MB".format(
        os.path.getsize(checkpoint_path) / (1024 ** 2)
    ))

/content/RT-DETR/ch 100%[===================>]  77.44M  39.5MB/s    in 2.0s    
Checkpoint: /content/RT-DETR/checkpoints/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth
Exists: True
Size: 77.44 MB


**الخلية 11 — إنشاء Config التدريب**

In [21]:
import os

config_path = "configs/rtdetrv2/rtdetrv2_adas_s.yml"

config = """__include__: [
  '../dataset/adas_detection.yml',
  '../runtime.yml',
  './include/dataloader.yml',
  './include/optimizer.yml',
  './include/rtdetrv2_r50vd.yml',
]

output_dir: ./output/rtdetrv2_adas_s

PResNet:
  depth: 18
  freeze_at: -1
  freeze_norm: False
  pretrained: True

HybridEncoder:
  in_channels: [128, 256, 512]
  hidden_dim: 256
  expansion: 0.5

RTDETRTransformerv2:
  num_layers: 3

epoches: 120
"""

with open(config_path, "w", encoding="utf-8") as f:
    f.write(config)

print("Created:", config_path)
print("\n--- RT-DETRv2-S ADAS Config ---\n")

with open(config_path, "r", encoding="utf-8") as f:
    print(f.read())

Created: configs/rtdetrv2/rtdetrv2_adas_s.yml

--- RT-DETRv2-S ADAS Config ---

__include__: [
  '../dataset/adas_detection.yml',
  '../runtime.yml',
  './include/dataloader.yml',
  './include/optimizer.yml',
  './include/rtdetrv2_r50vd.yml',
]

output_dir: ./output/rtdetrv2_adas_s

PResNet:
  depth: 18
  freeze_at: -1
  freeze_norm: False
  pretrained: True

HybridEncoder:
  in_channels: [128, 256, 512]
  hidden_dim: 256
  expansion: 0.5

RTDETRTransformerv2:
  num_layers: 3

epoches: 120



**الخلية 12 — Sanity Check**

In [22]:
from src.core import YAMLConfig

config_path = "configs/rtdetrv2/rtdetrv2_adas_s.yml"

cfg = YAMLConfig(config_path)

print("Configuration loaded successfully.")

print("\nTask:", cfg.yaml_cfg.get("task"))
print("Number of classes:", cfg.yaml_cfg.get("num_classes"))
print("Output directory:", cfg.yaml_cfg.get("output_dir"))

print("\nTrain images:", len(cfg.train_dataloader.dataset))
print("Validation images:", len(cfg.val_dataloader.dataset))

print("\n✓ Dataset and configuration loaded successfully.")

Configuration loaded successfully.

Task: detection
Number of classes: 14
Output directory: ./output/rtdetrv2_adas_s
building train_dataloader with batch_size=16...
loading annotations into memory...
Done (t=0.06s)
creating index...
index created!

Train images: 6191
building val_dataloader with batch_size=32...
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Validation images: 1767

✓ Dataset and configuration loaded successfully.


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


**الخلية 13 — بناء المودل فقط + فحص الـVRAM**

In [23]:
import torch

from src.core import YAMLConfig

config_path = "configs/rtdetrv2/rtdetrv2_adas_s.yml"
checkpoint_path = "/content/RT-DETR/checkpoints/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth"

cfg = YAMLConfig(config_path)

model = cfg.model

print("Model built successfully.")
print("Model type:", type(model).__name__)

if torch.cuda.is_available():
    model = model.cuda()
    print("\nGPU:", torch.cuda.get_device_name(0))

    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3

    print("VRAM allocated: {:.2f} GB".format(allocated))
    print("VRAM reserved: {:.2f} GB".format(reserved))

print("\nNumber of classes:", cfg.yaml_cfg["num_classes"])

Downloading: "https://github.com/lyuwenyu/storage/releases/download/v0.1/ResNet18_vd_pretrained_from_paddle.pth" to /root/.cache/torch/hub/checkpoints/ResNet18_vd_pretrained_from_paddle.pth


100%|██████████| 42.8M/42.8M [00:00<00:00, 159MB/s]


Load PResNet18 state_dict
Model built successfully.
Model type: RTDETR

GPU: Tesla T4
VRAM allocated: 0.08 GB
VRAM reserved: 0.09 GB

Number of classes: 14


**الخلية 14 — تحميل الـpretrained RT-DETRv2-S checkpoint**

In [24]:
import torch

checkpoint_path = "/content/RT-DETR/checkpoints/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu"
)

print("Checkpoint loaded successfully.")
print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    for key in checkpoint.keys():
        print(" -", key)

    if "ema" in checkpoint:
        print("\nEMA checkpoint found.")
    if "model" in checkpoint:
        print("Model checkpoint found.")

Checkpoint loaded successfully.
Checkpoint type: <class 'dict'>

Checkpoint keys:
 - ema

EMA checkpoint found.


**الخلية 15 — تحميل الـpretrained weights**

In [25]:
import torch

checkpoint_path = "/content/RT-DETR/checkpoints/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu"
)

ema_state = checkpoint["ema"]

if isinstance(ema_state, dict) and "module" in ema_state:
    pretrained_state = ema_state["module"]
else:
    pretrained_state = ema_state

model_state = model.state_dict()

compatible_state = {}
skipped_state = {}

for key, value in pretrained_state.items():
    if key in model_state and model_state[key].shape == value.shape:
        compatible_state[key] = value
    else:
        skipped_state[key] = value

model_state.update(compatible_state)
model.load_state_dict(model_state)

print("Pretrained weights loaded successfully.")

print("\nCompatible parameters loaded:", len(compatible_state))
print("Parameters skipped:", len(skipped_state))

print("\nSkipped parameters:")
for key in skipped_state:
    print(" -", key)

Pretrained weights loaded successfully.

Compatible parameters loaded: 529
Parameters skipped: 9

Skipped parameters:
 - decoder.denoising_class_embed.weight
 - decoder.enc_score_head.weight
 - decoder.enc_score_head.bias
 - decoder.dec_score_head.0.weight
 - decoder.dec_score_head.0.bias
 - decoder.dec_score_head.1.weight
 - decoder.dec_score_head.1.bias
 - decoder.dec_score_head.2.weight
 - decoder.dec_score_head.2.bias


**الخلية 16 — Forward + Backward صحيح**

In [4]:
%cd /content/RT-DETR/rtdetrv2_pytorch

path = "configs/dataset/adas_detection.yml"

content = """task: detection

evaluator:
  type: CocoEvaluator
  iou_types: ['bbox']

num_classes: 14
remap_mscoco_category: False

train_dataloader:
  type: DataLoader
  dataset:
    type: CocoDetection
    img_folder: /content/adas_coco_rtdetr/train
    ann_file: /content/adas_coco_rtdetr/train/_annotations.coco.json
    return_masks: False
    transforms:
      type: Compose
      ops: ~
  shuffle: True
  num_workers: 2
  drop_last: True
  collate_fn:
    type: BatchImageCollateFunction

val_dataloader:
  type: DataLoader
  dataset:
    type: CocoDetection
    img_folder: /content/adas_coco_rtdetr/valid
    ann_file: /content/adas_coco_rtdetr/valid/_annotations.coco.json
    return_masks: False
    transforms:
      type: Compose
      ops: ~
  shuffle: False
  num_workers: 2
  drop_last: False
  collate_fn:
    type: BatchImageCollateFunction
"""

with open(path, "w", encoding="utf-8") as f:
    f.write(content)

print("✓ adas_detection.yml updated successfully")
print()
print(content)

/content/RT-DETR/rtdetrv2_pytorch
✓ adas_detection.yml updated successfully

task: detection

evaluator:
  type: CocoEvaluator
  iou_types: ['bbox']

num_classes: 14
remap_mscoco_category: False

train_dataloader:
  type: DataLoader
  dataset:
    type: CocoDetection
    img_folder: /content/adas_coco_rtdetr/train
    ann_file: /content/adas_coco_rtdetr/train/_annotations.coco.json
    return_masks: False
    transforms:
      type: Compose
      ops: ~
  shuffle: True
  num_workers: 2
  drop_last: True
  collate_fn:
    type: BatchImageCollateFunction

val_dataloader:
  type: DataLoader
  dataset:
    type: CocoDetection
    img_folder: /content/adas_coco_rtdetr/valid
    ann_file: /content/adas_coco_rtdetr/valid/_annotations.coco.json
    return_masks: False
    transforms:
      type: Compose
      ops: ~
  shuffle: False
  num_workers: 2
  drop_last: False
  collate_fn:
    type: BatchImageCollateFunction



**🟦 الخلية 1 — فحص Labels في Dataset**

In [5]:
%cd /content/RT-DETR/rtdetrv2_pytorch

import json
import os

ann_path = "/content/adas_coco_rtdetr/train/_annotations.coco.json"

with open(ann_path, "r", encoding="utf-8") as f:
    data = json.load(f)

labels = sorted(set(
    ann["category_id"]
    for ann in data["annotations"]
))

print("=" * 80)
print("RT-DETR Dataset Label Check")
print("=" * 80)
print(f"Number of categories: {len(data['categories'])}")
print(f"Category IDs used: {labels}")
print(f"Minimum label: {min(labels)}")
print(f"Maximum label: {max(labels)}")
print("=" * 80)

assert labels == list(range(14)), \
    f"ERROR: Expected labels 0-13, but found {labels}"

print("✓ Dataset labels are valid: 0-13")
print("✓ Number of classes: 14")
print("✓ Safe to proceed with RT-DETRv2")

/content/RT-DETR/rtdetrv2_pytorch
RT-DETR Dataset Label Check
Number of categories: 14
Category IDs used: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
Minimum label: 0
Maximum label: 13
✓ Dataset labels are valid: 0-13
✓ Number of classes: 14
✓ Safe to proceed with RT-DETRv2


**🟦 الخلية 2 — فحص RT-DETRv2 + Dataset Configuration**

In [6]:
%cd /content/RT-DETR/rtdetrv2_pytorch

from src.core import YAMLConfig

cfg = YAMLConfig(
    "configs/rtdetrv2/rtdetrv2_adas_s.yml",
    resume=None
)

print("=" * 80)
print("RT-DETRv2 Configuration Check")
print("=" * 80)

print(f"Task: {cfg.yaml_cfg.get('task')}")
print(f"Number of classes: {cfg.yaml_cfg.get('num_classes')}")
print(f"Output directory: {cfg.yaml_cfg.get('output_dir')}")

print("=" * 80)
print("Dataset Configuration")
print("=" * 80)

train_dataset = cfg.train_dataloader.dataset
val_dataset = cfg.val_dataloader.dataset

print(f"Train image folder: {train_dataset.img_folder}")
print(f"Train annotation file: {train_dataset.ann_file}")
print(f"Validation image folder: {val_dataset.img_folder}")
print(f"Validation annotation file: {val_dataset.ann_file}")

print("=" * 80)
print("✓ Configuration loaded successfully")
print("✓ Number of classes: 14")
print("✓ Using remapped COCO dataset: 0-13")
print("=" * 80)

/content/RT-DETR/rtdetrv2_pytorch
RT-DETRv2 Configuration Check
Task: detection
Number of classes: 14
Output directory: ./output/rtdetrv2_adas_s
Dataset Configuration
building train_dataloader with batch_size=16...
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!
building val_dataloader with batch_size=32...
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Train image folder: /content/adas_coco_rtdetr/train
Train annotation file: /content/adas_coco_rtdetr/train/_annotations.coco.json
Validation image folder: /content/adas_coco_rtdetr/valid
Validation annotation file: /content/adas_coco_rtdetr/valid/_annotations.coco.json
✓ Configuration loaded successfully
✓ Number of classes: 14
✓ Using remapped COCO dataset: 0-13


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


**🟦 الخلية 3 — فحص Batch والـ Targets**

In [7]:
%cd /content/RT-DETR/rtdetrv2_pytorch

import torch

print("=" * 80)
print("RT-DETRv2 Batch and Target Check")
print("=" * 80)

batch = next(iter(cfg.train_dataloader))

images, targets = batch

print(f"Images shape: {images.shape}")
print(f"Number of targets: {len(targets)}")

all_labels = []

for i, target in enumerate(targets):
    labels = target["labels"]
    boxes = target["boxes"]

    all_labels.extend(labels.tolist())

    if i < 3:
        print(f"\nImage {i}:")
        print(f"  Labels shape: {labels.shape}")
        print(f"  Boxes shape: {boxes.shape}")
        print(f"  Labels: {labels.tolist()}")

all_labels = torch.tensor(all_labels)

print("\n" + "=" * 80)
print("Label Statistics")
print("=" * 80)

print(f"Minimum label: {all_labels.min().item()}")
print(f"Maximum label: {all_labels.max().item()}")
print(f"Unique labels: {sorted(all_labels.unique().tolist())}")

assert all_labels.min() >= 0
assert all_labels.max() < 14

print("=" * 80)
print("✓ Batch loaded successfully")
print("✓ All labels are within 0-13")
print("✓ Targets are valid for 14-class RT-DETRv2")
print("=" * 80)

/content/RT-DETR/rtdetrv2_pytorch
RT-DETRv2 Batch and Target Check


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Images shape: torch.Size([16, 3, 640, 640])
Number of targets: 16

Image 0:
  Labels shape: torch.Size([9])
  Boxes shape: torch.Size([9, 4])
  Labels: [3, 3, 3, 3, 3, 3, 3, 3, 3]

Image 1:
  Labels shape: torch.Size([0])
  Boxes shape: torch.Size([0, 4])
  Labels: []

Image 2:
  Labels shape: torch.Size([1])
  Boxes shape: torch.Size([1, 4])
  Labels: [4]

Label Statistics
Minimum label: 0
Maximum label: 13
Unique labels: [0, 1, 3, 4, 7, 13]
✓ Batch loaded successfully
✓ All labels are within 0-13
✓ Targets are valid for 14-class RT-DETRv2


**🟦 الخلية 4 — اختبار Forward + Loss + Backward**

In [8]:
%cd /content/RT-DETR/rtdetrv2_pytorch

import torch

print("=" * 80)
print("RT-DETRv2 Forward + Loss + Backward Test")
print("=" * 80)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")

model = cfg.model.to(device)
criterion = cfg.criterion

images, targets = batch

images = images.to(device)

for target in targets:
    for key in ["boxes", "labels", "image_id", "idx"]:
        if key in target and torch.is_tensor(target[key]):
            target[key] = target[key].to(device)

model.train()

print("Running forward pass...")

outputs = model(images, targets)

print("✓ Forward pass completed")

print("Calculating loss...")

loss_dict = criterion(outputs, targets)

loss = sum(loss_dict.values())

print("✓ Loss calculation completed")

print(f"Total loss: {loss.item():.6f}")

print("Running backward pass...")

loss.backward()

print("✓ Backward pass completed")

print("=" * 80)
print("✓ RT-DETRv2 Forward + Loss + Backward test PASSED")
print("✓ CUDA device-side assert is resolved")
print("=" * 80)

/content/RT-DETR/rtdetrv2_pytorch
RT-DETRv2 Forward + Loss + Backward Test
Device: cuda
Load PResNet18 state_dict
Running forward pass...
✓ Forward pass completed
Calculating loss...
✓ Loss calculation completed
Total loss: 28.700872
Running backward pass...
✓ Backward pass completed
✓ RT-DETRv2 Forward + Loss + Backward test PASSED
✓ CUDA device-side assert is resolved


**🟦 الخلية 5 — التحقق من الـ Pretrained Checkpoint**

In [9]:
%cd /content/RT-DETR/rtdetrv2_pytorch

import os
import torch

checkpoint_path = "/content/RT-DETR/checkpoints/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth"

print("=" * 80)
print("RT-DETRv2 Pretrained Checkpoint Check")
print("=" * 80)

print(f"Checkpoint: {checkpoint_path}")

assert os.path.exists(checkpoint_path), \
    f"Checkpoint not found: {checkpoint_path}"

file_size_mb = os.path.getsize(checkpoint_path) / (1024 ** 2)

print(f"File size: {file_size_mb:.2f} MB")

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print(f"Checkpoint type: {type(checkpoint).__name__}")
print(f"Checkpoint keys: {list(checkpoint.keys())}")

assert "ema" in checkpoint, "EMA weights not found in checkpoint"

print("=" * 80)
print("✓ Pretrained checkpoint exists")
print("✓ Checkpoint loaded successfully")
print("✓ EMA weights are available")
print("=" * 80)

/content/RT-DETR/rtdetrv2_pytorch
RT-DETRv2 Pretrained Checkpoint Check
Checkpoint: /content/RT-DETR/checkpoints/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth
File size: 77.44 MB
Checkpoint type: dict
Checkpoint keys: ['ema']
✓ Pretrained checkpoint exists
✓ Checkpoint loaded successfully
✓ EMA weights are available


**🟦 الخلية 6 — تحميل أوزان COCO في RT-DETRv2**

In [10]:
%cd /content/RT-DETR/rtdetrv2_pytorch

import torch

checkpoint_path = "/content/RT-DETR/checkpoints/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth"

print("=" * 80)
print("Loading RT-DETRv2 COCO Pretrained Weights")
print("=" * 80)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

state = checkpoint["ema"]["module"]

model_state = model.state_dict()

compatible = {}
skipped = {}

for key, value in state.items():
    if key in model_state:
        if model_state[key].shape == value.shape:
            compatible[key] = value
        else:
            skipped[key] = (
                tuple(value.shape),
                tuple(model_state[key].shape)
            )

print(f"Checkpoint tensors: {len(state)}")
print(f"Compatible tensors: {len(compatible)}")
print(f"Skipped tensors: {len(skipped)}")

model_state.update(compatible)
model.load_state_dict(model_state)

print("=" * 80)
print("✓ Compatible COCO weights loaded")
print("✓ Dataset-specific 14-class heads remain randomly initialized")
print("=" * 80)

if skipped:
    print("\nSkipped tensors due to shape mismatch:")
    for key, shapes in skipped.items():
        print(f"  {key}: {shapes[0]} → {shapes[1]}")

/content/RT-DETR/rtdetrv2_pytorch
Loading RT-DETRv2 COCO Pretrained Weights
Checkpoint tensors: 538
Compatible tensors: 529
Skipped tensors: 9
✓ Compatible COCO weights loaded
✓ Dataset-specific 14-class heads remain randomly initialized

Skipped tensors due to shape mismatch:
  decoder.denoising_class_embed.weight: (81, 256) → (15, 256)
  decoder.enc_score_head.weight: (80, 256) → (14, 256)
  decoder.enc_score_head.bias: (80,) → (14,)
  decoder.dec_score_head.0.weight: (80, 256) → (14, 256)
  decoder.dec_score_head.0.bias: (80,) → (14,)
  decoder.dec_score_head.1.weight: (80, 256) → (14, 256)
  decoder.dec_score_head.1.bias: (80,) → (14,)
  decoder.dec_score_head.2.weight: (80, 256) → (14, 256)
  decoder.dec_score_head.2.bias: (80,) → (14,)


**🟦 الخلية 7 — تجهيز إعدادات التدريب**

In [11]:
%cd /content/RT-DETR/rtdetrv2_pytorch

from pathlib import Path

source_config = Path("configs/rtdetrv2/rtdetrv2_adas_s.yml")
training_config = Path("configs/rtdetrv2/rtdetrv2_adas_baseline.yml")

content = source_config.read_text(encoding="utf-8")

content = content.replace(
    "output_dir: ./output/rtdetrv2_adas_s",
    "output_dir: ./output/rtdetrv2_adas_baseline"
)

training_config.write_text(content, encoding="utf-8")

print("=" * 80)
print("RT-DETRv2 Baseline Training Configuration")
print("=" * 80)

print(f"Source config: {source_config}")
print(f"Training config: {training_config}")
print()
print("Output directory:")
print("./output/rtdetrv2_adas_baseline")
print("=" * 80)
print("✓ Baseline configuration created")
print("✓ Original configuration preserved")
print("=" * 80)

/content/RT-DETR/rtdetrv2_pytorch
RT-DETRv2 Baseline Training Configuration
Source config: configs/rtdetrv2/rtdetrv2_adas_s.yml
Training config: configs/rtdetrv2/rtdetrv2_adas_baseline.yml

Output directory:
./output/rtdetrv2_adas_baseline
✓ Baseline configuration created
✓ Original configuration preserved


**🟦 الخلية 9 — بدء التدريب الفعلي**

In [14]:
%cd /content/RT-DETR/rtdetrv2_pytorch

!python tools/train.py \
    -c configs/rtdetrv2/rtdetrv2_adas_baseline.yml \
    -t /content/RT-DETR/checkpoints/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth \
    --use-amp

/content/RT-DETR/rtdetrv2_pytorch
Not init distributed mode.
cfg:  {'task': 'detection', '_model': None, '_postprocessor': None, '_criterion': None, '_optimizer': None, '_lr_scheduler': None, '_lr_warmup_scheduler': None, '_train_dataloader': None, '_val_dataloader': None, '_ema': None, '_scaler': None, '_train_dataset': None, '_val_dataset': None, '_collate_fn': None, '_evaluator': None, '_writer': None, 'num_workers': 0, 'batch_size': None, '_train_batch_size': None, '_val_batch_size': None, '_train_shuffle': None, '_val_shuffle': None, 'resume': None, 'tuning': '/content/RT-DETR/checkpoints/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth', 'epoches': 120, 'last_epoch': -1, 'use_amp': True, 'use_ema': True, 'ema_decay': 0.9999, 'ema_warmups': 2000, 'sync_bn': True, 'clip_max_norm': 0.1, 'find_unused_parameters': False, 'seed': None, 'print_freq': 100, 'checkpoint_freq': 1, 'output_dir': './output/rtdetrv2_adas_baseline', 'summary_dir': None, 'device': '', 'yaml_cfg': {'task': 'detection', 'e

**🟦 الخلية 10 — تخفيض Batch Size إلى 8**

In [15]:
%cd /content/RT-DETR/rtdetrv2_pytorch

from pathlib import Path

config_path = Path("configs/rtdetrv2/rtdetrv2_adas_baseline.yml")

content = config_path.read_text(encoding="utf-8")

content = content.replace(
    "total_batch_size: 16",
    "total_batch_size: 8"
)

config_path.write_text(content, encoding="utf-8")

print("=" * 80)
print("RT-DETRv2 Batch Size Adjustment")
print("=" * 80)
print("Training batch size: 16 → 8")
print("Validation batch size: 32")
print("=" * 80)
print("✓ Configuration updated")
print("✓ Ready to start training")
print("=" * 80)

/content/RT-DETR/rtdetrv2_pytorch
RT-DETRv2 Batch Size Adjustment
Training batch size: 16 → 8
Validation batch size: 32
✓ Configuration updated
✓ Ready to start training


**🟦 الخلية 11 — تشغيل التدريب**

In [16]:
%cd /content/RT-DETR/rtdetrv2_pytorch

!python tools/train.py \
    -c configs/rtdetrv2/rtdetrv2_adas_baseline.yml \
    -t /content/RT-DETR/checkpoints/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth \
    --use-amp

/content/RT-DETR/rtdetrv2_pytorch
Not init distributed mode.
cfg:  {'task': 'detection', '_model': None, '_postprocessor': None, '_criterion': None, '_optimizer': None, '_lr_scheduler': None, '_lr_warmup_scheduler': None, '_train_dataloader': None, '_val_dataloader': None, '_ema': None, '_scaler': None, '_train_dataset': None, '_val_dataset': None, '_collate_fn': None, '_evaluator': None, '_writer': None, 'num_workers': 0, 'batch_size': None, '_train_batch_size': None, '_val_batch_size': None, '_train_shuffle': None, '_val_shuffle': None, 'resume': None, 'tuning': '/content/RT-DETR/checkpoints/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth', 'epoches': 120, 'last_epoch': -1, 'use_amp': True, 'use_ema': True, 'ema_decay': 0.9999, 'ema_warmups': 2000, 'sync_bn': True, 'clip_max_norm': 0.1, 'find_unused_parameters': False, 'seed': None, 'print_freq': 100, 'checkpoint_freq': 1, 'output_dir': './output/rtdetrv2_adas_baseline', 'summary_dir': None, 'device': '', 'yaml_cfg': {'task': 'detection', 'e

**🟦 الخلية 10 — إصلاح Batch Size وتنظيف GPU**

In [17]:
%cd /content/RT-DETR/rtdetrv2_pytorch

from pathlib import Path
import torch
import gc

config_path = Path("configs/rtdetrv2/rtdetrv2_adas_baseline.yml")

content = config_path.read_text(encoding="utf-8")

content = content.replace(
    "    total_batch_size: 16",
    "    total_batch_size: 8"
)

config_path.write_text(content, encoding="utf-8")

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print("=" * 80)
print("RT-DETRv2 Training Fix")
print("=" * 80)
print("Training batch size: 8")
print("AMP: Enabled")
print("Epochs: 120")
print()
print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print("=" * 80)
print("✓ Batch size fixed")
print("✓ CUDA cache cleared")
print("✓ Ready to train")
print("=" * 80)

/content/RT-DETR/rtdetrv2_pytorch
RT-DETRv2 Training Fix
Training batch size: 8
AMP: Enabled
Epochs: 120

GPU allocated: 0.25 GB
GPU reserved:  1.30 GB
✓ Batch size fixed
✓ CUDA cache cleared
✓ Ready to train


**🟦 الخلية 11 — التدريب الفعلي**

In [18]:
%cd /content/RT-DETR/rtdetrv2_pytorch

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python tools/train.py \
    -c configs/rtdetrv2/rtdetrv2_adas_baseline.yml \
    -t /content/RT-DETR/checkpoints/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth \
    --use-amp

/content/RT-DETR/rtdetrv2_pytorch
Not init distributed mode.
cfg:  {'task': 'detection', '_model': None, '_postprocessor': None, '_criterion': None, '_optimizer': None, '_lr_scheduler': None, '_lr_warmup_scheduler': None, '_train_dataloader': None, '_val_dataloader': None, '_ema': None, '_scaler': None, '_train_dataset': None, '_val_dataset': None, '_collate_fn': None, '_evaluator': None, '_writer': None, 'num_workers': 0, 'batch_size': None, '_train_batch_size': None, '_val_batch_size': None, '_train_shuffle': None, '_val_shuffle': None, 'resume': None, 'tuning': '/content/RT-DETR/checkpoints/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth', 'epoches': 120, 'last_epoch': -1, 'use_amp': True, 'use_ema': True, 'ema_decay': 0.9999, 'ema_warmups': 2000, 'sync_bn': True, 'clip_max_norm': 0.1, 'find_unused_parameters': False, 'seed': None, 'print_freq': 100, 'checkpoint_freq': 1, 'output_dir': './output/rtdetrv2_adas_baseline', 'summary_dir': None, 'device': '', 'yaml_cfg': {'task': 'detection', 'e

In [19]:
import os

config_path = '/content/RT-DETR/rtdetrv2_pytorch/configs/rtdetrv2/rtdetrv2_adas_baseline.yml'
drive_output = '/content/drive/MyDrive/RT-DETR_Training/rtdetrv2_adas_baseline'

os.makedirs(drive_output, exist_ok=True)

with open(config_path, 'r') as f:
    config = f.read()

config = config.replace(
    'output_dir: ./output/rtdetrv2_adas_baseline',
    f'output_dir: {drive_output}'
)

with open(config_path, 'w') as f:
    f.write(config)

print("✓ Output directory changed")
print(f"✓ Every checkpoint will now be saved directly to:")
print(drive_output)

✓ Output directory changed
✓ Every checkpoint will now be saved directly to:
/content/drive/MyDrive/RT-DETR_Training/rtdetrv2_adas_baseline


In [20]:
import os

path = '/content/drive/MyDrive/RT-DETR_Training/rtdetrv2_adas_baseline'

checkpoints = []

for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith('.pth'):
            full_path = os.path.join(root, file)
            size_mb = os.path.getsize(full_path) / (1024 ** 2)
            checkpoints.append((full_path, size_mb))

checkpoints.sort(key=lambda x: os.path.getmtime(x[0]))

if checkpoints:
    print("Checkpoints found:\n")
    for i, (file, size) in enumerate(checkpoints, 1):
        print(f"{i}. {file} — {size:.2f} MB")

    latest = checkpoints[-1][0]
    print("\n✓ Checkpoint selected for resume:")
    print(latest)
else:
    print("❌ No .pth checkpoint found in Google Drive.")

❌ No .pth checkpoint found in Google Drive.


In [21]:

import os
import shutil

src = '/content/RT-DETR/rtdetrv2_pytorch/output/rtdetrv2_adas_baseline'
dst = '/content/drive/MyDrive/RT-DETR_Training/rtdetrv2_adas_baseline'

os.makedirs(dst, exist_ok=True)

pth_files = []

for root, dirs, files in os.walk(src):
    for file in files:
        if file.endswith('.pth'):
            full_path = os.path.join(root, file)
            pth_files.append(full_path)

if not pth_files:
    print("❌ No local checkpoint found.")
else:
    for file in pth_files:
        target = os.path.join(dst, os.path.basename(file))
        shutil.copy2(file, target)
        print(f"✓ Copied: {os.path.basename(file)}")

    print("\n✓ Checkpoint(s) are now in Google Drive.")

✓ Copied: last.pth
✓ Copied: checkpoint0000.pth
✓ Copied: checkpoint0001.pth
✓ Copied: checkpoint0002.pth
✓ Copied: best.pth
✓ Copied: latest.pth
✓ Copied: 000.pth

✓ Checkpoint(s) are now in Google Drive.


In [22]:
import torch
import os

path = '/content/drive/MyDrive/RT-DETR_Training/rtdetrv2_adas_baseline'

for name in ['last.pth', 'latest.pth', 'checkpoint0002.pth', 'best.pth']:
    file = os.path.join(path, name)

    if os.path.exists(file):
        ckpt = torch.load(file, map_location='cpu', weights_only=False)

        print(f"\n{name}")
        print(f"  epoch: {ckpt.get('epoch', 'N/A')}")
        print(f"  keys: {list(ckpt.keys())}")


last.pth
  epoch: N/A
  keys: ['date', 'last_epoch', 'model', 'criterion', 'postprocessor', 'ema', 'scaler', 'optimizer', 'lr_scheduler', 'lr_warmup_scheduler']

latest.pth
  epoch: N/A
  keys: ['params', 'counts', 'date', 'matched', 'precision', 'scores', 'recall', 'evaluations_size']

checkpoint0002.pth
  epoch: N/A
  keys: ['date', 'last_epoch', 'model', 'criterion', 'postprocessor', 'ema', 'scaler', 'optimizer', 'lr_scheduler', 'lr_warmup_scheduler']

best.pth
  epoch: N/A
  keys: ['date', 'last_epoch', 'model', 'criterion', 'postprocessor', 'ema', 'scaler', 'optimizer', 'lr_scheduler', 'lr_warmup_scheduler']


In [23]:
import torch
import os

path = '/content/drive/MyDrive/RT-DETR_Training/rtdetrv2_adas_baseline'

for name in ['last.pth', 'checkpoint0002.pth', 'best.pth']:
    file = os.path.join(path, name)

    if os.path.exists(file):
        ckpt = torch.load(file, map_location='cpu', weights_only=False)
        print(f"{name}: last_epoch = {ckpt['last_epoch']}")

last.pth: last_epoch = 2
checkpoint0002.pth: last_epoch = 2
best.pth: last_epoch = 2


In [ ]:
%cd /content/RT-DETR/rtdetrv2_pytorch

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python tools/train.py \
    -c configs/rtdetrv2/rtdetrv2_adas_baseline.yml \
    -r /content/drive/MyDrive/RT-DETR_Training/rtdetrv2_adas_baseline/last.pth \
    --use-amp

/content/RT-DETR/rtdetrv2_pytorch
Not init distributed mode.
cfg:  {'task': 'detection', '_model': None, '_postprocessor': None, '_criterion': None, '_optimizer': None, '_lr_scheduler': None, '_lr_warmup_scheduler': None, '_train_dataloader': None, '_val_dataloader': None, '_ema': None, '_scaler': None, '_train_dataset': None, '_val_dataset': None, '_collate_fn': None, '_evaluator': None, '_writer': None, 'num_workers': 0, 'batch_size': None, '_train_batch_size': None, '_val_batch_size': None, '_train_shuffle': None, '_val_shuffle': None, 'resume': '/content/drive/MyDrive/RT-DETR_Training/rtdetrv2_adas_baseline/last.pth', 'tuning': None, 'epoches': 120, 'last_epoch': -1, 'use_amp': True, 'use_ema': True, 'ema_decay': 0.9999, 'ema_warmups': 2000, 'sync_bn': True, 'clip_max_norm': 0.1, 'find_unused_parameters': False, 'seed': None, 'print_freq': 100, 'checkpoint_freq': 1, 'output_dir': '/content/drive/MyDrive/RT-DETR_Training/rtdetrv2_adas_baseline', 'summary_dir': None, 'device': '', 'y